## 🌍 Geo-Spatial & Regional Intelligence

This section focuses on understanding groundwater patterns at the district level.  
We analyze how risk, extraction, and utilization vary across different regions.

##### Import Libraries

In [20]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

##### Load Dataset

In [21]:
df = pd.read_csv("groundwater_ml_dataset_cleaned.csv")
df.head()

,state,district,annual_recharge,extractable_resource,annual_extraction,stage_of_development,category,extraction_ratio,utilization_rate,stress_level,risk_score,year
0,Himachal Pradesh,Himachal Pradesh,0.61,0.18,0.13,0.20,Safe,0.213115,0.722222,0.0020,0.374535,2024
1,Madhya Pradesh,Madhya Pradesh,27.00,1.68,0.17,7.04,Safe,0.006296,0.101190,0.0704,0.057075,2024
2,Andhra Pradesh,Alluri Sitharama Raju,43956.31,102516.48,2860.84,8890.60,Over Exploited,0.065084,0.027906,88.9060,17.818396,2024
3,Andhra Pradesh,Anakapalli,22443.69,38195.76,14423.44,6889.74,Over Exploited,0.642650,0.377619,68.8974,14.187588,2024
4,Andhra Pradesh,Ananthapuramu,40986.63,44150.14,1323.64,35512.65,Over Exploited,0.032294,0.029980,355.1265,71.050210,2024


#### 🎨 Color Theme Used

Soft and clean color palette:
- Light tones for better readability  
- Smooth gradients for numeric values  
- Consistent look across all charts  

### 1 State-wise District Risk (SLICER BAR CHART)
- District Risk Analysis with State Filter

- 💡 Insight:

User state select karega
Us state ke district-wise risk scores show honge
Easily identify high-risk districts inside each state

In [36]:
# STEP 1: Prepare Data
# -------------------------------
district_risk = df.groupby(['state','district'])['risk_score'].mean().reset_index()

# -------------------------------
# STEP 2: Initial State
# -------------------------------
first_state = district_risk['state'].unique()[0]
init_df = district_risk[district_risk['state'] == first_state]

# -------------------------------
# STEP 3: Create Figure
# -------------------------------
fig = go.Figure()

fig.add_trace(go.Bar(
    x=init_df['district'],
    y=init_df['risk_score'],
    marker=dict(
        color=init_df['risk_score'],   # 🔥 gradient based on value
        colorscale='Tealgrn'
    )
))

# -------------------------------
# STEP 4: Dropdown (SLICER)
# -------------------------------
buttons = []

for state in district_risk['state'].unique():
    temp = district_risk[district_risk['state'] == state]

    buttons.append(dict(
        label=state,
        method='update',
        args=[{
            'x': [temp['district']],
            'y': [temp['risk_score']],
            'marker.color': [temp['risk_score']]  # 🔥 update colors too
        }]
    ))

# -------------------------------
# STEP 5: Layout
# -------------------------------
fig.update_layout(
    title='District-wise Risk Score (State Filter)',
    template='plotly_white',
    title_x=0.5,
    xaxis_title='District',
    yaxis_title='Risk Score',
    updatemenus=[dict(
        buttons=buttons,
        direction='down',
        x=0,
        y=1.15,
        xanchor='left',
        yanchor='top'
    )]
)

# -------------------------------
# STEP 6: Show
# -------------------------------
fig.show()

### 2. Recharge vs Extraction

In [31]:
fig2 = px.scatter(
    df,
    x='annual_recharge',
    y='annual_extraction',
    color='risk_score',
    size='risk_score',
    hover_name='district',
    title='Recharge vs Extraction (District Level)',
    color_continuous_scale=px.colors.sequential.Blugrn
)

fig2.update_layout(template='plotly_white', title_x=0.5)
fig2.show()

### 📊 3 Extraction Ratio vs Risk

- This chart shows how extraction ratio affects groundwater risk.
- Higher extraction ratio usually leads to higher risk.

In [24]:
fig4 = px.scatter(
    df,
    x='extraction_ratio',
    y='risk_score',
    color='risk_score',
    hover_name='district',
    title='Extraction Ratio vs Risk',
    color_continuous_scale=px.colors.sequential.Blugrn_r
)

fig4.update_layout(template='plotly_white', title_x=0.5)
fig4.show()

### 📊 4 Bubble Chart (Extraction vs Risk)

This chart combines three factors:
- Extraction (x-axis)
- Risk (y-axis)
- Utilization (bubble size)

It gives a complete view of groundwater stress.

In [25]:
fig7 = px.scatter(
    df,
    x='annual_extraction',
    y='risk_score',
    size='utilization_rate',
    color='risk_score',
    hover_name='district',
    title='Extraction vs Risk (Bubble Chart)',
    color_continuous_scale=px.colors.sequential.Blugrn_r
)

fig7.update_layout(template='plotly_white', title_x=0.5)
fig7.show()

### 5. Top 10 States by Extraction

- Shows states with highest groundwater extraction.

In [26]:
top_ext = df.groupby('state')['annual_extraction'].sum().reset_index()
top_ext = top_ext.sort_values('annual_extraction', ascending=False).head(10)

fig1 = px.bar(
    top_ext,
    x='state',
    y='annual_extraction',
    color='annual_extraction',
    color_continuous_scale=px.colors.sequential.Blugrn_r,
    title='Top 10 States by Extraction'
)

fig1.update_layout(template='plotly_white', title_x=0.5)
fig1.show()

### 6 Top 10 States by Recharge

- Shows states with highest groundwater recharge.

In [27]:
top_rec = df.groupby('state')['annual_recharge'].sum().reset_index()
top_rec = top_rec.sort_values('annual_recharge', ascending=False).head(10)

fig2 = px.treemap(
    top_rec,
    path=['state'],
    values='annual_recharge',
    color='annual_recharge',
    color_continuous_scale=px.colors.sequential.Blugrn_r,
    title='Top 10 States by Recharge'
)

fig2.update_layout(title_x=0.5)
fig2.show()

### 7. State-wise Average Risk Score

- Shows average groundwater risk in each state.

In [39]:
avg_risk_state = df.groupby('state')['risk_score'].mean().reset_index()

fig3 = px.bar(
    avg_risk_state,
    x='state',
    y='risk_score',
    color='risk_score',
    color_continuous_scale=px.colors.sequential.Tealgrn,  # ✅ Blue-Green
    title='State-wise Avg Risk Score'
)

fig3.update_layout(
    template='plotly_white',
    title_x=0.5
)

fig3.show()

### 8 Top 10 High Risk Districts

- Highlights districts with highest risk scores.

In [29]:
top_districts = df.sort_values('risk_score', ascending=False).head(10)

fig4 = px.treemap(
    top_districts,
    path=['district'],
    values='risk_score',
    color='risk_score',
    color_continuous_scale=px.colors.sequential.Blugrn,
    title='Top 10 High Risk Districts'
)

fig4.update_layout(title_x=0.5)
fig4.show()

### 9. State-wise Risk Sunburst 
Hierarchical Risk Flow
- 💡 Insight:

Shows how risk flows from State → District → Category

In [30]:

sun_df = df.groupby(['state','district','category'])['risk_score'].mean().reset_index()

fig = px.sunburst(
    sun_df,
    path=['state','district','category'],
    values='risk_score',
    color='risk_score',
    color_continuous_scale='Tealgrn',
    title='Risk Distribution Sunburst'
)

fig.update_layout(template='plotly_white', title_x=0.5)

fig.show()

### 🔍 Key Takeaways

- Some districts show very high groundwater risk.
- High extraction and utilization increase risk levels.
- Water balance helps identify surplus and deficit regions.
- Proper monitoring is needed for sustainable water use.